[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FieldQuantum/fieldqkit/blob/main/examples/demo_vqe_h2_4q.ipynb)

# VQE 教程：用 4 比特重建 H₂ 分子的解离曲线

本教程用变分量子本征求解器（VQE）计算氢分子 H₂ 在不同键长下的基态能量，
画出它的**势能曲线（potential energy surface）**，并和精确的 FCI 结果对比。

我们会走完一条完整的研究路径：

1. **准备哈密顿量**：从预先生成的数据文件中读取不同键长 $R$ 下、经 Jordan–Wigner 映射得到的 4 比特 H₂ 哈密顿量。
2. **设计 ansatz**：对比两种试探态——通用的 *Hardware-Efficient* ansatz 与受化学启发的 *UCC 风格* ansatz。
3. **模拟器扫描**：在 `Simulator` 上扫描键长，得到两种 ansatz 的解离曲线。
4. **真实硬件 + 误差缓解**：把 UCC ansatz 放到真实芯片上运行，并叠加 ZNE、读出纠错、Clifford 拟合等误差缓解手段。

> 数据文件位于 `examples/data/`，每个 `h2_R{R}_angstrom_sto-3g.json` 包含该键长下的哈密顿量项、常数项和 FCI 参考能量。默认全程使用 `Simulator`，便于稳定复现。

In [ ]:
# 在 Google Colab 上运行请先执行本单元格安装依赖（本地已安装可跳过）
%pip install -q "fieldqkit[sim]"

## 1) 配置与数据目录

In [ ]:
from pathlib import Path
import json
import math

# Locate the bundled experiment data directory (works locally and on Colab).
# The JSON files live in examples/data/ next to this notebook.
DATA_DIR = next(
    (p for p in (Path('data'), Path('examples/data'), Path('fieldqkit/examples/data'))
     if p.exists()),
    Path('data'),
)

# Geometry / chemistry config
R_UNIT = 'angstrom'
BASIS = 'sto-3g'
MULTIPLICITY = 1
CHARGE = 0

# VQE config
cfg = {
    'layers': 3,
    'shots': 4096,
    'max_iters': 50,
    'learning_rate': 0.2,
    'seed': 7,
    'gradient_method': 'autograd',
    'prefer_chips': 'Simulator',
    'shift': math.pi / 2,
}

## 2) 构造 UCC 风格的化学 ansatz

对 H₂（STO-3G、Jordan–Wigner 映射后 4 比特）来说，Hartree–Fock 参考态是 $|1100\rangle$——
即用 `X` 门把 0、1 号比特置为 $|1\rangle$，代表两个被占据的自旋轨道。

在此基础上加一个**单激发-双激发**式的相关项，这里用符号参数 `theta` 写成一个 Pauli 演化门
$e^{-i\,\theta\,(Y_0 Y_1 Y_2 X_3)/2}$。

In [ ]:
from fieldqkit import QuantumHardwareClient
from fieldqkit.algorithms import VQERunner
from fieldqkit.circuit import QuantumCircuit

def build_h2_ucc_style_symbolic_ansatz() -> QuantumCircuit:
    qc = QuantumCircuit(4)

    qc = QuantumCircuit(4)

    # -------------------------
    # Hartree–Fock reference
    # |1100>
    # -------------------------

    qc.x(0)
    qc.x(1)

    qc.pauli_evolution("theta", "Y0 Y1 Y2 X3")
    return qc

custom_ansatz_qc = build_h2_ucc_style_symbolic_ansatz()
symbolic_params = sorted(
    [k for k, v in custom_ansatz_qc.params_value.items() if isinstance(k, str) and isinstance(v, str)]
)
print('Total gates in custom H2-UCC-style ansatz:', len(custom_ansatz_qc.gates))

## 3) 在模拟器上扫描键长

In [ ]:
from __future__ import annotations

import os
import json
from pathlib import Path

from fieldqkit import QuantumHardwareClient
from fieldqkit.algorithms import VQERunner
import numpy as np

R_range = np.arange(0.5, 3.01, 0.3)
energy_he = []
energy_ucc = []
for r, R in enumerate(R_range):
    json_path = DATA_DIR / f'h2_R{R:.1f}_angstrom_sto-3g.json'

    data = json.loads(json_path.read_text(encoding='utf-8'))

    h2_constant = float(data['constant'])
    h2_4q_terms = [(float(c), str(obs)) for c, obs in data['terms']]
    nqubits = int(data['nqubits'])
    fci_energy = float(data['fci_energy'])

    print('Loaded from:', json_path)

    client = QuantumHardwareClient()
    runner = VQERunner(
        client=client,
        shots=cfg['shots'],
        max_iters=cfg['max_iters'],
        learning_rate=cfg['learning_rate'],
        gradient_method=cfg['gradient_method'],
        seed=cfg['seed'],
    )

    result = runner.run_model(
        name=f'h2_4q_R{R:.1f}_he',
        num_qubits=nqubits,
        model='custom',
        hamiltonian=h2_4q_terms,
        prefer_chips=cfg['prefer_chips'],
        ansatz='hardwareefficient',
    )

    best_total_energy = h2_constant + result.best_energy
    abs_error_fci = abs(best_total_energy - fci_energy)
    energy_he.append({
        'R': R,
        'estimated_total_energy': best_total_energy,
    })

    result = runner.run_model(
        name=f'h2_4q_R{R:.1f}_ucc',
        num_qubits=nqubits,
        model='custom',
        hamiltonian=h2_4q_terms,
        prefer_chips=cfg['prefer_chips'],
        ansatz='custom',
        custom_ansatz_circuit=custom_ansatz_qc,
        init_params=[0.01]*len(symbolic_params)
        
    )

    best_total_energy = h2_constant + result.best_energy
    abs_error_fci = abs(best_total_energy - fci_energy)
    energy_ucc.append({
        'R': R,
        'estimated_total_energy': best_total_energy,
    })

In [ ]:
from matplotlib import pyplot as plt

R_values = [res['R'] for res in energy_he]
estimated_energies_he = [res['estimated_total_energy'] for res in energy_he]
estimated_energies_ucc = [res['estimated_total_energy'] for res in energy_ucc]
R_range = np.arange(0.5, 3.01, 0.1)
fci_energies = []
for R in R_range:
    json_path = DATA_DIR / f'h2_R{R:.1f}_angstrom_sto-3g.json'
    data = json.loads(json_path.read_text(encoding='utf-8'))
    fci_energies.append(float(data['fci_energy']))
plt.figure(figsize=(10, 6))
plt.scatter(R_values, estimated_energies_he, marker='o', label='Hardware-Efficient Ansatz')
plt.scatter(R_values, estimated_energies_ucc, marker='s', label='Unitary Coupled Cluster Ansatz')
plt.plot(R_range, fci_energies, 'k-', label='FCI Energy')
plt.xlabel('Interatomic Distance R (angstrom)')
plt.ylabel('Energy (Hartree)')
plt.legend()
plt.title('H2 Molecule Energy vs Interatomic Distance')
plt.tick_params(right=True, top=True)
plt.tight_layout()
plt.show()

## 4) 真实硬件 + 误差缓解

最后把 UCC ansatz 放到真实芯片上跑（这里 `prefer_chips='Baihua'`）。真实硬件有噪声，
直接测得的能量会明显偏离 FCI，因此我们打开框架内置的几种**误差缓解**手段：`zne=True`、`readout_mitigation=True`、`clifford_fitting=True`。
此外把 `gradient_method` 换成 `'parameter-shift'`（真实硬件上无法自动微分），并把 `max_iters`
调小。

In [ ]:
cfg['prefer_chips'] = 'Baihua'
cfg['zne'] = True
cfg['readout_mitigation'] = True
cfg['clifford_fitting'] = True
cfg['clifford_fitting_num_samples'] = 8
cfg['max_iters'] = 10
cfg['gradient_method'] = 'parameter-shift'
energy_ucc_exp = []
R_range = np.arange(0.5, 3.01, 0.3)
for r, R in enumerate(R_range):
    json_path = DATA_DIR / f'h2_R{R:.1f}_angstrom_sto-3g.json'
    data = json.loads(json_path.read_text(encoding='utf-8'))

    h2_constant = float(data['constant'])
    h2_4q_terms = [(float(c), str(obs)) for c, obs in data['terms']]
    nqubits = int(data['nqubits'])
    fci_energy = float(data['fci_energy'])

    print('Loaded from:', json_path)

    client = QuantumHardwareClient()
    runner = VQERunner(
        client=client,
        shots=cfg['shots'],
        max_iters=cfg['max_iters'],
        learning_rate=cfg['learning_rate'],
        gradient_method=cfg['gradient_method'],
        seed=cfg['seed'],
        zne=cfg['zne'],
        readout_mitigation=cfg['readout_mitigation'],
        shift=cfg['shift'],
        clifford_fitting=cfg['clifford_fitting'],
        clifford_fitting_num_samples=cfg['clifford_fitting_num_samples'],
    )

    result = runner.run_model(
        name=f'h2_4q_R{R:.1f}_ucc',
        num_qubits=nqubits,
        model='custom',
        hamiltonian=h2_4q_terms,
        prefer_chips=cfg['prefer_chips'],
        ansatz='custom',
        custom_ansatz_circuit=custom_ansatz_qc,
        init_params=[0.01]*len(symbolic_params)
    )

    best_total_energy = h2_constant + result.best_energy
    abs_error_fci = abs(best_total_energy - fci_energy)
    energy_ucc_exp.append({
        'R': R,
        'estimated_total_energy': best_total_energy,
    })

In [ ]:
R_values = [res['R'] for res in energy_ucc_exp]
estimated_energies_ucc = [res['estimated_total_energy'] for res in energy_ucc_exp]
R_range = np.arange(0.5, 3.01, 0.1)
fci_energies = []
for R in R_range:
    json_path = DATA_DIR / f'h2_R{R:.1f}_angstrom_sto-3g.json'
    data = json.loads(json_path.read_text(encoding='utf-8'))
    fci_energies.append(float(data['fci_energy']))
plt.figure(figsize=(10, 6))
plt.scatter(R_values, estimated_energies_ucc, marker='s', label='Unitary Coupled Cluster Ansatz')
plt.plot(R_range, fci_energies, 'k-', label='FCI Energy')
plt.xlabel('Interatomic Distance R (angstrom)')
plt.ylabel('Energy (Hartree)')
plt.legend()
plt.title('H2 Molecule Energy vs Interatomic Distance')
plt.tick_params(right=True, top=True)
plt.tight_layout()
plt.show()

## 5) 小结与下一步

本教程从哈密顿量数据出发，完整走通了「读数据 → 设计 ansatz → 模拟器扫描 → 真机 + 误差缓解 → 画解离曲线」。

可以继续尝试：

- **加密键长网格**：把 `R_range` 步长从 0.3 改成 0.1，得到更平滑的曲线（注意运行时间也会变长）。
- **丰富 ansatz**：在 UCC 风格线路里再加一项激发（如再加一个 `pauli_evolution`），看是否进一步贴近 FCI。
- **调超参数**：增大 `max_iters`、调整 `learning_rate`，观察收敛行为。
- **量化误差**：记录每个 $R$ 的 `abs_error_fci`，画出误差随键长的变化，定量评估 ansatz 与误差缓解的效果。
- **换芯片/开关缓解项**：单独打开/关闭 `zne`、`readout_mitigation`、`clifford_fitting`，比较各自的贡献。